In [1]:
"""
Group Relative Policy Optimization (GRPO)

RLVR (reinforcement learning through verifiable rewards) determines what learning signal
is available, and GRPO determines how that signal is used to update the model weights.

A common used policy gradient algorithm for LLMs is Proximal Policy Optimization (PPO)
GRPO is more resource friendly than PPO because it doesn't require
a separate value model to estimate the value function.

GRPO learning signal iks from relative comparisons within a group of sampled responses
"""

"\nGroup Relative Policy Optimization (GRPO)\n\nRLVR (reinforcement learning through verifiable rewards) determines what learning signal\nis available, and GRPO determines how that signal is used to update the model weights.\n\nA common used policy gradient algorithm for LLMs is Proximal Policy Optimization (PPO)\nGRPO is more resource friendly than PPO because it doesn't require\na separate value model to estimate the value function.\n\nGRPO learning signal iks from relative comparisons within a group of sampled responses\n"

In [2]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)
print(response)

 47 47


In [4]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    # if already downloaded
    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [5]:
from pprint import pprint
pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

In [6]:
# Typical batched input_ids
input_ids = torch.randint(0, 50000, (4, 256))   # batch=4, seq_len=256

print(input_ids)
print(len(input_ids))        # → 4          (batch size)
print(input_ids.numel())     # → 1024       (total tokens = 4 × 256)
print(input_ids.shape[1])    # → 256        (sequence length)

tensor([[ 4866, 33054, 38805,  ..., 32271,  1705, 12195],
        [26584, 15093, 17803,  ..., 30627, 22836, 39823],
        [46354, 45237, 13219,  ..., 15546, 26202, 13188],
        [10715, 17507,  3446,  ..., 16872, 34173, 18903]])
4
1024
256


In [7]:
a = torch.tensor([[1, 2],
                  [3, 4]])          # shape [2, 2]

b = torch.tensor([[5, 6],
                  [7, 8]])          # shape [2, 2]

result = torch.cat([a, b], dim=0)
# tensor([[1, 2],
#         [3, 4],
#         [5, 6],
#         [7, 8]])                  # shape [4, 2]
result

tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])

In [8]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter

@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
    )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        logits = logits.float()
        
        if temperature and temperature != 1.0:
            logits = logits / temperature

        # Optional safety clamp (MAC)
        # logits = torch.clamp(logits, min=-50.0, max=50.0)

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)

        # Extra guard (very useful on MPS)
        # probas = torch.nan_to_num(probas, nan=0.0, posinf=0.0, neginf=0.0)
        # probas = torch.clamp(probas, min=0.0)
        # probas = probas / probas.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break

        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids, torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )

    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [9]:
# Fixed sample_response for MPS: cast logits to float32 before softmax to prevent
# NaN from bfloat16/float16 overflow on MPS, and use [:, -1] (last position) for
# correct autoregressive generation.
@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
    )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        logits = logits.float()  # prevent NaN from low-precision overflow on MPS

        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)

        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break

        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids, torch.tensor(generated, device=device, dtype=input_ids.dtype)]
    )

    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [10]:
torch.manual_seed(0)
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    device=device,
    max_new_tokens=512,
    temperature=0.9,
    top_p=0.9,
)

print(answer_text)

 47<|endoftext|>


In [11]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

In [12]:
from reasoning_from_scratch.ch03 import extract_final_candidate, grade_answer

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(answer_text, fallback=None)
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [13]:
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



In [14]:
# advantage values directly scale the gradients during the policy update
rewards = torch.tensor(rollout_rewards, device=device)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
print(rewards)
print(advantages)

tensor([1., 1., 0., 0.])
tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


In [15]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    return torch.mean(next_token_logps).item()

In [16]:
avg_logprob_val = avg_logprob_answer(
    model, tokenizer,
    prompt=prompt,
    answer=answer_text,
    device=device
)
print(avg_logprob_val)

-2.21875


In [17]:
"""
Sequence-level (what GRPO does)
"How much better/worse was this complete answer compared to the other complete answers I generated?"

Token-level (the alternative)
"Given the words so far, how good/likely was generating this particular next word?"
"""
sequence_logprob_val = avg_logprob_val * len(tokenizer.encode(answer_text))
print(sequence_logprob_val)

-8.875


In [18]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-8.9066, grad_fn=<SumBackward0>)


In [19]:
"""
same as above `sequence_logprob_draft` but more optimized, easier to read, and faster to execute
"""
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(1, token_ids[1:].unsqueeze(-1)).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-8.9066, grad_fn=<SumBackward0>)


In [20]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer: {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer: \boxed{83}
Logprob: -8.1092

Answer: The correct answer is \boxed{83}
Logprob: -19.7631

Answer: The final answer is 83
Logprob: -16.4600

Answer: We get \boxed{38}
Logprob: -23.1289



In [21]:
"""
PyTorch optimizers are defined to minimize a loss, objectives that are naturally written
as maximization problems must have its sign flipped.
"""
logps = torch.stack(rollout_logps)
pg_loss = -(advantages.detach() * logps).mean()
print(logps)
print(pg_loss)

tensor([ -8.1092, -19.7631, -16.4600, -23.1289], grad_fn=<StackBackward0>)
tensor(-2.5363, grad_fn=<NegBackward0>)


In [22]:
"""
Combining all GRPO
"""
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_rewards, samples = [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    # Phase 1: generate all rollouts with no_grad (sampling only)
    rollout_data = []
    for _ in range(num_rollouts):
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        reward = reward_rlvr(text, example["answer"])
        rollout_data.append((token_ids, prompt_len))
        roll_rewards.append(reward)
        samples.append({
            "text": text,
            "reward": reward,
            "gen_len": token_ids.numel() - prompt_len,
        })

    if was_training:
        model.train()

    # Phase 2: advantages from rewards
    rewards = torch.tensor(roll_rewards, device=device)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Phase 3: one rollout at a time — backward and free each graph before
    # computing the next. Keeps only one computation graph in memory at a time
    # instead of holding all num_rollouts graphs simultaneously (÷num_rollouts
    # peak activation memory). Gradient accumulation gives identical math to
    # calling backward on the stacked mean loss.
    total_loss = 0.0
    for i, (token_ids, prompt_len) in enumerate(rollout_data):
        logp = sequence_logprob(model, token_ids, prompt_len)
        loss_i = -(advantages[i].detach() * logp) / num_rollouts
        loss_i.backward()
        total_loss += loss_i.item()
        del logp, loss_i, token_ids
        if device.type == "mps":
            torch.mps.synchronize()
            torch.mps.empty_cache()

    return {
        "loss": total_loss,
        "pg_loss": total_loss,
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
    }

In [23]:
torch.manual_seed(123)
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': 0.0,
 'pg_loss': 0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      '\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.\n'
                      '\n'
                      ' 

In [24]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,
):
    if steps is None:
        steps = len(math_data)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0

    if csv_log_path is None:
        timestamp = time.strftime("%Y%%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        for step in range(steps):
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # backward() is called inside compute_grpo_loss, one rollout at a
            # time, so only one computation graph is in memory at once.
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if device.type == "mps":
                torch.mps.synchronize()
                torch.mps.empty_cache()

            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(sample["gen_len"] for sample in stats["samples"])
            avg_response_len = step_tokens / len(stats["samples"]) if stats["samples"] else 0.0
            append_csv_metrics(
                csv_log_path, current_step, steps,
                stats["loss"], reward_avg, avg_response_len,
            )
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model

def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = checkpoint_dir / f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path

def append_csv_metrics(
    csv_log_path, step_idx, total_steps,
    loss, reward_avg, avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [25]:
device = get_device()

# MPS (Apple Silicon) has known numerical issues with bfloat16 (NaN in matmuls,
# softmax with -inf masking, etc.). Force float32 so all computations are stable.
if device.type == "mps":
    model.cfg["dtype"] = torch.float32
    model.to(device, dtype=torch.float32)
else:
    model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=0.0000 reward_avg=0.000 avg_resp_len=6.0
[Step 2/50] loss=0.0000 reward_avg=0.000 avg_resp_len=130.0
[Step 3/50] loss=0.0000 reward_avg=0.000 avg_resp_len=64.5
[Step 4/50] loss=0.0000 reward_avg=0.000 avg_resp_len=17.5
[Step 5/50] loss=0.0000 reward_avg=0.000 avg_resp_len=43.0
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth
[Step 6/50] loss=0.0000 reward_avg=0.000 avg_resp_len=256.0


RuntimeError: MPS backend out of memory (MPS allocated: 13.59 GiB, other allocations: 6.43 GiB, max allowed: 20.13 GiB). Tried to allocate 203.44 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).